In [1]:
import torch

In [2]:
class KVCache:
  def __init__(self):
    self.k = None
    self.v = None

  def update(self,new_k,new_v):
    if self.k is None:
       self.k = new_k
       self.v = new_v
    else:
       self.k = torch.cat([self.k,new_k],dim=2)
       self.v = torch.cat([self.v,new_v],dim=2)

  def clear(self):
    self.k = None
    self.v = None

In [5]:
def incremental_attention(q_new, k_new, v_new, cache):
  cache.update(k_new,v_new)
  k_all = cache.k
  v_all = cache.v

  d_k = q_new.size(-1)
  result = q_new @ k_all.transpose(-2,-1)
  score = result // (d_k ** 0.5)
  weight = torch.softmax(score,dim=-1)
  output = weight @ v_all
  return output

In [6]:
class KVCache:
  def __init__(self):
    self.k = None
    self.v = None

  def update(self,new_k,new_v):
    if self.k is None:
       self.k = new_k
       self.v = new_v
    else:
       self.k = torch.cat([self.k,new_k],dim=2)
       self.v = torch.cat([self.v,new_v],dim=2)

  def clear(self):
    self.k = None
    self.v = None

def incremental_attention(q_new, k_new, v_new, cache):
  cache.update(k_new,v_new)
  k_all = cache.k
  v_all = cache.v

  d_k = q_new.size(-1)
  result = q_new @ k_all.transpose(-2,-1)
  score = result // (d_k ** 0.5)
  weight = torch.softmax(score,dim=-1)
  output = weight @ v_all
  return output


B, H, Dh = 2, 4, 16
cache = KVCache()
for step in range(3):
    # 模拟当前步生成的 q, k, v (形状全是 [2, 4, 1, 16])
    q_new = torch.randn(B, H, 1, Dh)
    k_new = torch.randn(B, H, 1, Dh)
    v_new = torch.randn(B, H, 1, Dh)

    # 计算增量 Attention
    out = incremental_attention(q_new, k_new, v_new, cache)

    print(f"Step {step+1}:")
    print(f"  当前 k_new 形状: {k_new.shape}")
    print(f"  缓存 k_all 形状: {cache.k.shape}  <-- 长度逐渐累加")
    print(f"  输出 out 形状:   {out.shape}")

Step 1:
  当前 k_new 形状: torch.Size([2, 4, 1, 16])
  缓存 k_all 形状: torch.Size([2, 4, 1, 16])  <-- 长度逐渐累加
  输出 out 形状:   torch.Size([2, 4, 1, 16])
Step 2:
  当前 k_new 形状: torch.Size([2, 4, 1, 16])
  缓存 k_all 形状: torch.Size([2, 4, 2, 16])  <-- 长度逐渐累加
  输出 out 形状:   torch.Size([2, 4, 1, 16])
Step 3:
  当前 k_new 形状: torch.Size([2, 4, 1, 16])
  缓存 k_all 形状: torch.Size([2, 4, 3, 16])  <-- 长度逐渐累加
  输出 out 形状:   torch.Size([2, 4, 1, 16])
